# Qwen3.5-0.8B LoRA v2 — Colab launcher
GitHub carries source code and synthetic datasets. Google Drive receives checkpoints directly, so a disconnected free Colab session can resume without ZIP uploads.

In [ ]:
REPO_URL = "https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git"
BRANCH = "main"
PROJECT_DIR = "/content/local-slm-lab"
DRIVE_OUTPUT = "/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v2"
assert REPO_URL.startswith("https://github.com/"), "Set a valid GitHub URL"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

For a public GitHub repository, no token is needed. For a private repository, add a Colab secret named `GITHUB_TOKEN` with read access and enable notebook access to it. The token is sent as a temporary Git header and is not written into `.git/config`.

In [ ]:
import base64
import os
import pathlib
import subprocess

git_env = os.environ.copy()
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    encoded = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {encoded}",
    })

project = pathlib.Path(PROJECT_DIR)
if (project / ".git").is_dir():
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True, env=git_env)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH], check=True, env=git_env)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR], check=True, env=git_env)
print(subprocess.run(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{PROJECT_DIR}/training/requirements.txt"], check=True)
subprocess.run([sys.executable, "scripts/build-corpus-v2.py"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "scripts/verify-corpus-v2.py"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "-m", "unittest", "tests.test_corpus_v2", "tests.test_training_helpers", "-v"], cwd=PROJECT_DIR, check=True)

In [ ]:
import pathlib
import subprocess
import sys
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/local-slm-lab")
DRIVE_OUTPUT = globals().get("DRIVE_OUTPUT", "/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v2")
pathlib.Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)
train_command = [
    sys.executable, "training/train_lora.py",
    "--train", "corpus-v2/sft/train.jsonl",
    "--validation", "corpus-v2/sft/validation.jsonl",
    "--output", DRIVE_OUTPUT,
    "--epochs", "4",
    "--early-stopping-patience", "1",
    "--max-length", "3072",
    "--resume-from-checkpoint", "auto",
]
print("Starting or resuming:", " ".join(train_command))
subprocess.run(train_command, cwd=PROJECT_DIR, check=True)

In [ ]:
from pathlib import Path
run_dir = Path(DRIVE_OUTPUT)
print("Best adapter:", run_dir / "best-adapter")
print("Evaluation:", (run_dir / "evaluation.json").read_text() if (run_dir / "evaluation.json").exists() else "training not complete")
print("Checkpoints:", [path.name for path in sorted(run_dir.glob("checkpoint-*"))])